# LMP spread - per-node shadow-price exposure

Does each node's exposure to the binding constraints beat the PJM-wide shadow-price totals? Same GNN as `models/model.ipynb`, two input sets per fold:
- `base`: the 9 node features + 6 calendar features (15 inputs)
- `exposure`: base + `da_exposure`, `rt_exposure_prev_day` (17 inputs)

Facility-to-node weights are fit per fold on its training hours (`lmp_model/exposure.py`). Scored on the original 1,505 nodes (`role != 'constraint'`). Every run is logged to the `gridcast-lmp-spread` MLflow experiment.

In [ ]:
from functools import lru_cache
import time

from loguru import logger
import mlflow
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch_geometric.nn import SAGEConv

from lmp_model import exposure
from lmp_model.config import PROCESSED_DATA_DIR
from lmp_model.dataset import CALENDAR_FEATURES, load_cache, load_nodes
from lmp_model.features import fit_scaler, scale
from lmp_model.graph import build_edges, edge_index
from lmp_model.splits import split_masks, walk_forward_folds

folds = walk_forward_folds()
nodes = load_nodes()
pnode_ids = nodes.index
scored = (nodes["role"] != "constraint").to_numpy()
data = load_cache()
times = pd.DatetimeIndex(data["times"], tz="UTC")
calendar = data["system"][:, : len(CALENDAR_FEATURES)]
loss_mask = data["exists"] & ~np.isnan(data["y"]) & ~np.isnan(data["x"]).any(axis=-1)

Exposure inputs on the window cache's hours and nodes. Rerun after `window.npz` is rebuilt.

In [ ]:
exposure.build_cache(times, pnode_ids)

In [ ]:
shadow = exposure.load_cache()
assert (shadow["times"] == data["times"]).all()
{k: v.shape for k, v in shadow.items()}

In [ ]:
torch.set_num_threads(4)
HOURS_PER_BATCH = 16
Y_SCALE = 20.0
params = {
    "hidden": 64, "node_dim": 16, "layers": 2, "lr": 1e-3, "weight_decay": 1e-5,
    "hours_per_batch": HOURS_PER_BATCH, "max_epochs": 30, "patience": 5, "seed": 0,
    "y_scale": Y_SCALE, "top_facilities": exposure.TOP_FACILITIES, "ridge_alpha": exposure.RIDGE_ALPHA,
}
mlflow.set_experiment("gridcast-lmp-spread")


class SpreadGNN(nn.Module):
    def __init__(self, n_nodes, n_in, hidden=64, node_dim=16, layers=2):
        super().__init__()
        self.node_emb = nn.Embedding(n_nodes, node_dim)
        self.inp = nn.Linear(n_in + node_dim, hidden)
        self.convs = nn.ModuleList(SAGEConv(hidden, hidden) for _ in range(layers))
        self.out = nn.Linear(hidden, 1)

    def forward(self, x, edges):
        hours, nodes, _ = x.shape
        h = self.inp(torch.cat([x, self.node_emb.weight.expand(hours, -1, -1)], dim=-1)).relu()
        h = h.reshape(hours * nodes, -1)
        for conv in self.convs:
            h = h + conv(h, edges).relu()
        return self.out(h).reshape(hours, nodes)


def fold_blocks(fold, variant):
    """Scaled inputs per split: node features (+ exposure, with weights fit on this fold's
    training hours) and calendar, broadcast to every node. The loss mask is the same for both
    variants: the exposure columns' NaN (first 48 hours) are filled with 0 like other inputs."""
    masks = split_masks(data["times"], fold)
    x = data["x"]
    if variant == "exposure":
        top, weights = exposure.fit_weights(shadow, masks["train"])
        x = np.concatenate([x, exposure.exposure_features(shadow, top, weights)], axis=-1)
    x_scaler = fit_scaler(x[masks["train"]])
    cal_scaler = fit_scaler(calendar[masks["train"]])
    blocks = {}
    for name, m in masks.items():
        xs = np.nan_to_num(scale(x[m], x_scaler)).astype("float32")
        cs = np.nan_to_num(scale(calendar[m], cal_scaler)).astype("float32")
        inputs = np.concatenate([xs, np.broadcast_to(cs[:, None, :], (*xs.shape[:2], cs.shape[1]))], axis=-1)
        blocks[name] = {"x": torch.from_numpy(inputs), "y": data["y"][m], "mask": loss_mask[m]}
    return blocks


def evaluate(pred, block):
    """MAE on the original 1,505 nodes; spikes are the top 1% |spread| among those cells."""
    y = block["y"]
    m = block["mask"] & scored[None, :]
    spike = m & (np.abs(y) >= np.quantile(np.abs(y[m]), 0.99))
    err = np.abs(pred - y)
    return {"mae": float(err[m].mean()), "spike_mae": float(err[spike].mean())}

In [ ]:
def train_gnn(fold_i, variant):
    fold = folds[fold_i]
    blocks = fold_blocks(fold, variant)
    ei = torch.from_numpy(edge_index(build_edges(fold.train_start, fold.val_start), pnode_ids)).long()
    n_nodes = len(pnode_ids)
    batch_edges = lru_cache(lambda hours: torch.cat([ei + i * n_nodes for i in range(hours)], dim=1))

    @torch.no_grad()
    def predict(model, split):
        model.eval()
        x = blocks[split]["x"]
        out = [model(x[i : i + HOURS_PER_BATCH], batch_edges(len(x[i : i + HOURS_PER_BATCH]))) for i in range(0, len(x), HOURS_PER_BATCH)]
        model.train()
        return torch.cat(out).numpy() * Y_SCALE

    x, mask = blocks["train"]["x"], torch.from_numpy(blocks["train"]["mask"])
    y = torch.from_numpy(np.nan_to_num(blocks["train"]["y"]) / Y_SCALE)
    n_features = x.shape[-1]
    torch.manual_seed(params["seed"])
    model = SpreadGNN(n_nodes, n_features, params["hidden"], params["node_dim"], params["layers"])
    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    run_name = f"gnn_{variant}_full_{pd.Timestamp.now():%Y%m%d}"
    with mlflow.start_run(run_name=run_name) as run:
        out = PROCESSED_DATA_DIR / "runs" / run.info.run_id
        out.mkdir(parents=True)
        mlflow.set_tags({"fold": fold_i, "feature_set": variant, "graph": "full", "model": "sage_gnn", "local_artifacts": str(out)})
        mlflow.log_params({
            **params, "train_start": fold.train_start, "val_start": fold.val_start,
            "test_start": fold.test_start, "test_end": fold.test_end, "n_features": n_features,
            "n_nodes": n_nodes, "n_scored_nodes": int(scored.sum()), "n_edges_undirected": ei.shape[1] // 2,
        })
        best_val, best_epoch, stale = float("inf"), 0, 0
        for epoch in range(params["max_epochs"]):
            start = time.time()
            order = torch.randperm(len(x))
            total, n_batches = 0.0, len(x) // HOURS_PER_BATCH
            for b in range(n_batches):
                idx = order[b * HOURS_PER_BATCH : (b + 1) * HOURS_PER_BATCH]
                pred = model(x[idx], batch_edges(len(idx)))
                m = mask[idx]
                loss = nn.functional.huber_loss(pred[m], y[idx][m], delta=1.0)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                total += loss.item()
            val = evaluate(predict(model, "val"), blocks["val"])
            mlflow.log_metrics({"train_loss": total / n_batches, "val_mae": val["mae"], "val_spike_mae": val["spike_mae"], "epoch_seconds": time.time() - start}, step=epoch)
            logger.info(f"fold {fold_i} {variant} epoch {epoch}: val_mae {val['mae']:.2f}, val_spike_mae {val['spike_mae']:.1f}")
            if val["mae"] < best_val:
                best_val, best_epoch, stale = val["mae"], epoch, 0
                torch.save(model.state_dict(), out / "model.pt")
            else:
                stale += 1
                if stale >= params["patience"]:
                    break

        model.load_state_dict(torch.load(out / "model.pt"))
        val = evaluate(predict(model, "val"), blocks["val"])
        test = evaluate(predict(model, "test"), blocks["test"])
        zero_val = evaluate(np.zeros_like(blocks["val"]["y"]), blocks["val"])
        zero_test = evaluate(np.zeros_like(blocks["test"]["y"]), blocks["test"])
        result = {
            "fold": fold_i, "variant": variant, "n_inputs": n_features, "best_epoch": best_epoch,
            "val_mae": val["mae"], "zero_val_mae": zero_val["mae"],
            "test_mae": test["mae"], "test_spike_mae": test["spike_mae"],
            "zero_test_mae": zero_test["mae"], "zero_test_spike_mae": zero_test["spike_mae"],
            "test_skill": 1 - test["mae"] / zero_test["mae"],
        }
        mlflow.log_metrics({k: v for k, v in result.items() if k not in ("fold", "variant")})
    return result

Fold 0, both input sets (~20 min each on the Mac Mini).

In [ ]:
RESULTS_CSV = PROCESSED_DATA_DIR / "runs" / "exposure_comparison.csv"
results = [train_gnn(0, variant) for variant in ["base", "exposure"]]
pd.DataFrame(results).to_csv(RESULTS_CSV, index=False)
pd.DataFrame(results)

Folds 1-3.

In [ ]:
for fold_i in range(1, len(folds)):
    for variant in ["base", "exposure"]:
        results.append(train_gnn(fold_i, variant))
        pd.DataFrame(results).to_csv(RESULTS_CSV, index=False)
pd.DataFrame(results).pivot(index="fold", columns="variant", values=["val_mae", "test_mae", "test_spike_mae", "test_skill"]).round(3)